### Complex predicion and selectivity screening with Boltz-2

#### Objective
Co-fold each candidate against the two receptor assemblies:the βγc target and the α counter-target. <br>
Determine whether any shows confident target engagement together with α discrimination. Also, determine whether the metrics chosen for that judgement can tell a real protein from a shuffled one.

#### Methodology
Each candidate is run through Boltz-2 in two panels: <br>
chain A plus IL-2Rβ and γc (ternary), and chain A plus IL-2Rα (alpha), at 518 and 322 residues respectively.

Three controls are run through the identical pipeline. Native IL-2 is the positive control for the ternary panel and, since it binds α, also the positive control for engagement in the alpha panel. Neo-2/15 is the reference for what a selective molecule looks like. A composition-matched scrambled sequence is the negative control, and is the most important of the three: a metric that cannot distinguish a real protein from its own shuffled sequence cannot be used to rank designs.

#### Limitations
One seed and one diffusion sample per job

In [ ]:
# Stage 5 — Boltz-2 installation
#
# Why this is not just `pip install boltz`:
#
#   Colab runs Python 3.13. Boltz declares requires_python "<3.13,>=3.10", so
#   plain pip walks backwards through all 14 releases looking for one that
#   permits 3.13, finds none, and grinds for an hour before failing.
#
#   The bound is real but indirect. The Boltz wheel is py3-none-any — pure
#   Python, no compiled extensions. What excludes 3.13 is its pinned
#   dependency set: numpy<2.0, scipy==1.13.1, gemmi==0.6.5, biopython==1.84
#   all predate Python 3.13 and carry C extensions with no cp313 wheels.
#
#   Fix: install dependencies UNPINNED so pip picks 3.13-compatible versions,
#   then install Boltz with --no-deps and the Python bound overridden.
#
#   torch is deliberately NOT in DEPS. Colab ships a CUDA-linked build, and
#   `pip install torch` can replace it with a CPU-only wheel that installs
#   cleanly then fails at the first GPU call.
#
#   Trade-off: the environment no longer matches the package's own pins, so
#   "Boltz 2.2.1" does not fully specify this run. pip_freeze.txt does.
#
#   Supply chain: all from PyPI, where pip verifies each artifact's hash
#   against the index. Stronger than the RFdiffusion weights in Stage 2,
#   which have no published reference digest.
#
#   Run BEFORE mounting Drive.

import os, sys, subprocess, importlib, json

BOLTZ_VERSION = "2.2.1"
CUDA_INDEX = "https://download.pytorch.org/whl/cu124"

# wandb is a required dependency and the only one that transmits data by
# design. Disabled before anything imports it.
os.environ["WANDB_MODE"] = "offline"
os.environ["WANDB_DISABLED"] = "true"

def sh(args, quiet=True):
    print("$", " ".join(args))
    return subprocess.run(args + (["-q"] if quiet else []), check=False)

def torch_cuda_on_disk():
    """Read torch's CUDA status from disk, not from the imported module."""
    r = subprocess.run(
        ["python", "-c", "import torch; print(torch.version.cuda or 'NONE')"],
        capture_output=True, text=True)
    return r.stdout.strip()

def boltz_works():
    try:
        importlib.invalidate_caches()
        import boltz.main                                    # noqa: F401
        return subprocess.run(["boltz", "predict", "--help"],
                              capture_output=True, text=True).returncode == 0
    except Exception:
        return False

print(f"python {sys.version_info.major}.{sys.version_info.minor}")

import torch
if not torch.cuda.is_available():
    raise RuntimeError(
        "torch has no CUDA. Repair:\n"
        f"  pip install --force-reinstall torch --index-url {CUDA_INDEX}\n"
        "then Runtime -> Restart session.")
print(f"torch {torch.__version__} | cuda {torch.version.cuda} | "
      f"{torch.cuda.get_device_name(0)} "
      f"({torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB)")

if boltz_works():
    print("\nboltz already installed and functional — skipping")
else:
    DEPS = ["numpy", "scipy", "pandas", "scikit-learn", "numba",
            "rdkit", "biopython", "gemmi", "modelcif",
            "chembl_structure_pipeline",
            "hydra-core", "pytorch-lightning", "fairscale", "dm-tree",
            "einops", "einx", "mashumaro", "click", "pyyaml",
            "requests", "types-requests", "wandb"]          # no torch
    sh(["pip", "install"] + DEPS)
    sh(["pip", "install", f"boltz=={BOLTZ_VERSION}",
        "--no-deps", "--ignore-requires-python"])

    cuda = torch_cuda_on_disk()
    if cuda == "NONE":
        raise RuntimeError(
            "a dependency replaced torch with a CPU build. Repair:\n"
            f"  pip install --force-reinstall torch --index-url {CUDA_INDEX}\n"
            "then Runtime -> Restart session and re-run.")
    print(f"torch still CUDA-enabled on disk: {cuda}")

    if not boltz_works():
        raise RuntimeError(
            "boltz still not functional — see the error above. Fallback: "
            "Chai-1 (`pip install chai_lab`) performs the same co-folding "
            "task; Stage 5's logic is model-agnostic.")
    print("\nboltz installed and functional")

frozen = subprocess.run(["pip", "freeze"], capture_output=True,
                        text=True).stdout
versions = {p: next((l.split("==")[1] for l in frozen.splitlines()
                     if l.lower().startswith(p + "==")), None)
            for p in ["boltz", "numpy", "scipy", "torch", "numba",
                      "gemmi", "biopython", "rdkit", "pytorch-lightning"]}
print("\nresolved versions:")
for k, v in versions.items():
    print(f"  {k:20s} {v}")

BOLTZ_ENV = {
    "boltz_version": BOLTZ_VERSION,
    "python": f"{sys.version_info.major}.{sys.version_info.minor}",
    "install_method": "deps unpinned (torch excluded), "
                      "boltz --no-deps --ignore-requires-python",
    "declared_requires_python": "<3.13,>=3.10",
    "wandb": "disabled (offline)",
    "resolved": versions,
    "gpu": torch.cuda.get_device_name(0),
}
print("\n*** mount Drive in the next cell ***")

python 3.13
torch 2.11.0+cu130 | cuda 13.0 | NVIDIA L4 (24 GB)
$ pip install numpy scipy pandas scikit-learn numba rdkit biopython gemmi modelcif chembl_structure_pipeline hydra-core pytorch-lightning fairscale dm-tree einops einx mashumaro click pyyaml requests types-requests wandb
$ pip install boltz==2.2.1 --no-deps --ignore-requires-python
torch still CUDA-enabled on disk: 13.0


[12:32:23] Initializing Normalizer



boltz installed and functional

resolved versions:
  boltz                2.2.1
  numpy                2.1.3
  scipy                1.16.3
  torch                2.11.0+cu130
  numba                0.61.2
  gemmi                0.7.5
  biopython            1.88
  rdkit                2026.3.6
  pytorch-lightning    2.6.6

*** mount Drive in the next cell ***


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

PROJ = "/content/drive/MyDrive/il2_mimetic"
OUT4, OUT5 = f"{PROJ}/stage4_fold", f"{PROJ}/stage5_complex"
for d in ["", "/inputs", "/preds"]:
    os.makedirs(OUT5 + d, exist_ok=True)

with open(f"{OUT5}/pip_freeze.txt", "w") as fh:
    fh.write(frozen)
with open(f"{OUT5}/boltz_env.json", "w") as fh:
    json.dump(BOLTZ_ENV, fh, indent=2)
print("provenance written to", OUT5)

Mounted at /content/drive
provenance written to /content/drive/MyDrive/il2_mimetic/stage5_complex


In [ ]:
import os
p = f"{OUT4}/stage5_candidates_diverse.tsv"
print(p, "exists:", os.path.exists(p))

/content/drive/MyDrive/il2_mimetic/stage4_fold/stage5_candidates_diverse.tsv exists: False


In [ ]:
# Reduce 12 candidates to 6 on structural diversity rather than on rank,
# by greedy max-min over the pairwise TM matrix
import os, itertools
import numpy as np, pandas as pd

OUTT = f"{PROJ}/analysis_tm"
K = 6

tm_path = f"{OUTT}/candidates_tm_matrix.tsv"
assert os.path.exists(tm_path), (
    f"{tm_path} missing — re-run the TM-score cell in the analysis notebook")

tm = pd.read_csv(tm_path, sep="\t", index_col=0)
ids = list(tm.index)
M = tm.values.astype(float)
print(f"loaded {len(ids)}x{len(ids)} TM matrix")

def pick_diverse(M, k):
    """Greedy max-min: repeatedly add the candidate whose closest
    already-selected neighbour is the most distant."""
    n = len(M)
    Mx = M.copy(); np.fill_diagonal(Mx, 2.0)
    i, j = np.unravel_index(np.argmin(Mx), Mx.shape)   # most dissimilar pair
    sel = [int(i), int(j)]
    while len(sel) < k:
        rest = [x for x in range(n) if x not in sel]
        sel.append(min(rest, key=lambda x: max(M[x, s] for s in sel)))
    return sel

idx = pick_diverse(M, K)
keep = [ids[i] for i in idx]

full = pd.read_csv(f"{OUT4}/stage5_candidates.tsv", sep="\t")
sub  = full[full.seq_id.isin(keep)].copy()
sub.to_csv(f"{OUT4}/stage5_candidates_diverse.tsv", sep="\t", index=False)

print(f"\nselected {len(sub)}:")
print(sub[["seq_id", "arm", "variant", "length", "plddt",
           "sc_rmsd", "motif_rmsd"]].round(3).to_string(index=False))

off_all = M[np.triu_indices(len(M), 1)]
off_sub = M[np.ix_(idx, idx)][np.triu_indices(K, 1)]
print(f"\npairwise TM — all 12 : min {off_all.min():.3f} "
      f"median {np.median(off_all):.3f} max {off_all.max():.3f}")
print(f"pairwise TM — subset  : min {off_sub.min():.3f} "
      f"median {np.median(off_sub):.3f} max {off_sub.max():.3f}")
print(f"\nby arm    : {sub.arm.value_counts().to_dict()}")
print(f"by variant: {sub.variant.value_counts().to_dict()}")
print(f"\nwritten: {OUT4}/stage5_candidates_diverse.tsv")

loaded 12x12 TM matrix

selected 6:
                      seq_id         arm     variant  length  plddt  sc_rmsd  motif_rmsd
     pad0_len100_15_fixAll_4      fixAll pad0_len100     105  92.46    0.619       0.300
 pad0_len100_2_fixContacts_3 fixContacts pad0_len100     115  92.01    0.552       0.317
      pad2_len100_7_fixAll_3      fixAll pad2_len100      91  92.84    0.647       0.353
     pad0_len100_22_fixAll_0      fixAll pad0_len100     107  92.76    0.624       0.395
pad0_len100_37_fixContacts_2 fixContacts pad0_len100     111  92.80    0.513       0.477
pad0_len100_13_fixContacts_3 fixContacts pad0_len100      98  93.25    0.650       0.464

pairwise TM — all 12 : min 0.598 median 0.892 max 0.979
pairwise TM — subset  : min 0.598 median 0.891 max 0.921

by arm    : {'fixAll': 3, 'fixContacts': 3}
by variant: {'pad0_len100': 5, 'pad2_len100': 1}

written: /content/drive/MyDrive/il2_mimetic/stage4_fold/stage5_candidates_diverse.tsv


In [ ]:
# Build the 18 FASTA inputs. Receptor chains carry server MSAs. Designed
# and de novo chains carry explicitly empty MSAs, because a sequence with
# no homologs cannot have a meaningful alignment and the server would
# return noise for one
import os, json, random
import pandas as pd
from Bio import SeqIO

with open(f"{PROJ}/stage1_motif/params.json") as fh:
    CHAIN = json.load(fh)["chains"]

def seqres(p):
    return {r.id.split(":")[-1]: str(r.seq)
            for r in SeqIO.parse(p, "pdb-seqres")}

s      = seqres(f"{PROJ}/2B5I.pdb")
IL2Rb  = s[CHAIN["IL2Rb"]]
GC     = s[CHAIN["gc"]]
IL2Ra  = s[CHAIN["IL2Ra"]]
IL2    = s[CHAIN["IL2"]]
NEO    = seqres(f"{PROJ}/stage3_seqs/benchmark/6DG6.pdb")["A"]

cand = pd.read_csv(f"{OUT4}/stage5_candidates_diverse.tsv", sep="\t")
seqs = pd.read_csv(f"{PROJ}/stage3_seqs/sequences.tsv", sep="\t")
if "sequence" not in cand.columns:
    cand = cand.merge(seqs[["seq_id", "sequence"]], on="seq_id")

random.seed(0)
SCRAM = "".join(random.sample(cand.sequence.iloc[0], len(cand.sequence.iloc[0])))

# Boltz FASTA:  >CHAIN|protein        -> MSA computed by the server
#               >CHAIN|protein|empty  -> no MSA (correct for de novo designs)
def write_fasta(name, ligand, ligand_msa=False):
    rows = [("A", ligand, ligand_msa)]
    if name.endswith("_ternary"):
        rows += [("B", IL2Rb, True), ("C", GC, True)]
    else:
        rows += [("B", IL2Ra, True)]
    p = f"{OUT5}/inputs/{name}.fasta"
    with open(p, "w") as fh:
        for cid, seq, msa in rows:
            fh.write(f">{cid}|protein\n{seq}\n" if msa
                     else f">{cid}|protein|empty\n{seq}\n")
    return p

RUNS = {}
for label, seq, msa in [("IL2", IL2, True),      # natural, has homologs
                        ("Neo", NEO, False),     # de novo, none to find
                        ("scram", SCRAM, False)]:
    for panel in ("ternary", "alpha"):
        RUNS[f"ctrl_{label}_{panel}"] = write_fasta(
            f"ctrl_{label}_{panel}", seq, msa)

for _, r in cand.iterrows():
    for panel in ("ternary", "alpha"):
        RUNS[f"{r.seq_id}_{panel}"] = write_fasta(
            f"{r.seq_id}_{panel}", r.sequence, ligand_msa=False)

n_ctrl = sum("ctrl" in k for k in RUNS)
print(f"{len(RUNS)} runs: {n_ctrl} controls, {len(RUNS)-n_ctrl} design")
print(f"ternary size: {len(IL2Rb)+len(GC)+len(cand.sequence.iloc[0])} residues")
print(f"alpha   size: {len(IL2Ra)+len(cand.sequence.iloc[0])} residues")
print("\ncontrol runs:", [k for k in RUNS if k.startswith("ctrl")])
print("\nexample input file:")
print(open(RUNS["ctrl_IL2_ternary"]).read()[:300])

18 runs: 6 controls, 12 design
ternary size: 518 residues
alpha   size: 322 residues

control runs: ['ctrl_IL2_ternary', 'ctrl_IL2_alpha', 'ctrl_Neo_ternary', 'ctrl_Neo_alpha', 'ctrl_scram_ternary', 'ctrl_scram_alpha']

example input file:
>A|protein
APTSSSTKKTQLQLEHLLLDLQMILNGINNYKNPKLTRMLTFKFYMPKKATELKHLQCLEEELKPLEEVLNLAQSKNFHLRPRDLISNINVIVLELKGSETTFMCEYADETATIVEFLNRWITFCQSIISTLT
>B|protein
AVQGTSQFTCFYNSRAQISCVWSQDGALQDTSCQVHAWPDRRRWQQTCELLPVSQASWACNLILGAPDSQKLTTVDIVTLRVLCREGVRWRVMAIQDFKPFENLRLMAPISLQVVHVETHRCNISWEISQASHYFERHLEFEAR


In [ ]:
# First prediction attempt:fails loading the checkpoint. torch >= 2.6
# unpickles with weights_only=True, and Boltz-2's checkpoint embeds its
# training config as an omegaconf object the restricted unpickler
# refuses
import subprocess, time, glob

test = "ctrl_IL2_ternary"
t0 = time.time()
r = subprocess.run([
    "boltz", "predict", RUNS[test],
    "--out_dir", f"{OUT5}/preds",
    "--output_format", "pdb",
    "--seed", "0",
    "--accelerator", "gpu",
    "--use_msa_server",
    "--override",
], capture_output=True, text=True)
dt = time.time() - t0

print(f"exit {r.returncode} in {dt/60:.1f} min")
if r.returncode != 0:
    print(r.stdout[-2000:])
    print("--- STDERR ---")
    print(r.stderr[-3000:])
else:
    print(f"projected for {len(RUNS)} runs: {dt*len(RUNS)/3600:.1f} h")
    for f in sorted(glob.glob(f"{OUT5}/preds/**/*", recursive=True))[:15]:
        print("  ", f.replace(OUT5, ""))

exit 1 in 4.8 min
MSA server enabled: https://api.colabfold.com
MSA server authentication: no credentials provided
Extracting the CCD data to /root/.boltz/mols. This may take a bit of time. You may change the cache directory with the --cache flag.
Checking input data.
Processing 1 inputs with 1 threads.
Generating MSA for /content/drive/MyDrive/il2_mimetic/stage5_complex/inputs/ctrl_IL2_ternary.fasta with 3 protein entities.
Calling MSA server for target ctrl_IL2_ternary with 3 sequences
MSA server URL: https://api.colabfold.com
MSA pairing strategy: greedy
No authentication provided for MSA server
Running structure prediction for 1 input.

--- STDERR ---
/click/core.py", line 2032, in invoke
    return _process_result(sub_ctx.command.invoke(sub_ctx))
                           ~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^

... [22 lines trimmed] ...
  File "/usr/local/lib/python3.13/dist-packages/lightning_fabric/utilities/cloud_io.py", line 73, in _load
    return torch.load(
           ~~~~~~~~~~

In [ ]:
# Fix 1
import site, os, textwrap

sp = site.getsitepackages()[0]
path = os.path.join(sp, "sitecustomize.py")

code = textwrap.dedent('''
    # Allow only the omegaconf config classes that Boltz checkpoints contain.
    # torch>=2.6 defaults to weights_only=True because pickle can execute
    # arbitrary code. We narrow the allowlist rather than disabling the check.
    try:
        import torch.serialization as _ts
        _safe = []
        try:
            from omegaconf.dictconfig import DictConfig
            from omegaconf.listconfig import ListConfig
            from omegaconf.base import ContainerMetadata, Metadata
            from omegaconf.nodes import AnyNode, ValueNode
            _safe += [DictConfig, ListConfig, ContainerMetadata, Metadata,
                      AnyNode, ValueNode]
        except Exception:
            pass
        import typing, collections
        _safe += [typing.Any, collections.defaultdict, dict, list, int, str]
        _ts.add_safe_globals(_safe)
    except Exception:
        pass
''')

with open(path, "w") as fh:
    fh.write(code)
print("written:", path)

# verify it loads in a fresh interpreter without breaking anything
import subprocess
r = subprocess.run(["python", "-c",
                    "import torch, sitecustomize; print('sitecustomize OK')"],
                   capture_output=True, text=True)
print(r.stdout or r.stderr[-800:])

written: /usr/local/lib/python3.13/dist-packages/sitecustomize.py
sitecustomize OK



In [ ]:
# Retry: identical failure. The file was written and imports cleanly, so
# the allowlist is not being applied at interpreter startup
import os, subprocess, time, pathlib

PROJ = "/content/drive/MyDrive/il2_mimetic"
S5   = f"{PROJ}/stage5_complex"
IN   = f"{S5}/inputs"
OUT  = f"{S5}/preds"
os.makedirs(OUT, exist_ok=True)

# guard: the CUDA build must still be the one on disk
r = subprocess.run(["python","-c","import torch; print(torch.version.cuda or 'NONE')"],
                   capture_output=True, text=True)
cuda = r.stdout.strip()
assert cuda != "NONE", f"torch on disk has no CUDA ({cuda}) — reinstall before running"
print("torch cuda:", cuda)

job = "ctrl_IL2_ternary"
fa  = f"{IN}/{job}.fasta"
assert os.path.exists(fa), fa

t0 = time.time()
p = subprocess.run(
    ["boltz","predict", fa,
     "--out_dir", OUT,
     "--output_format","pdb",
     "--seed","0",
     "--accelerator","gpu",
     "--use_msa_server",
     "--override"],
    capture_output=True, text=True)
dt = (time.time()-t0)/60

print(f"exit {p.returncode}   {dt:.1f} min")
print(p.stdout[-2500:])
if p.returncode != 0:
    print("---- stderr ----")
    print(p.stderr[-3500:])

torch cuda: 13.0
exit 1   0.2 min
MSA server enabled: https://api.colabfold.com
MSA server authentication: no credentials provided
Checking input data.
All inputs are already processed.
Processing 0 inputs with 0 threads.
Running structure prediction for 1 input.

---- stderr ----
ts automatically to the Lightning Experiments platform.
Traceback (most recent call last):
  File "/usr/local/bin/boltz", line 8, in <module>
    sys.exit(cli())
             ~~~^^
  File "/usr/local/lib/python3.13/dist-packages/click/core.py", line 1631, in __call__
    return self.main(*args, **kwargs)
           ~~~~~~~~~^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/click/core.py", line 1552, in main
    rv = self.invoke(ctx)

... [25 lines trimmed] ...
  File "/usr/local/lib/python3.13/dist-packages/lightning_fabric/utilities/cloud_io.py", line 73, in _load
    return torch.load(
           ~~~~~~~~~~^
        f,
        ^^
        map_location=map_location,  # type: ignore[arg-type]
 

In [ ]:
# Diagnosis: sitecustomize resolves to Debian's own copy in /usr/lib,
# which precedes dist-packages on sys.path. The file written above is
# shadowed and never executed
import subprocess
print(subprocess.run(["python","-c",
  "import sitecustomize; print(sitecustomize.__file__)"],
  capture_output=True, text=True).stdout)

/usr/lib/python3.13/sitecustomize.py



In [ ]:
# 2nd Fix
import os, subprocess, textwrap

os.makedirs("/content/tools", exist_ok=True)
WRAPPER = "/content/tools/boltz_run.py"

with open(WRAPPER, "w") as fh:
    fh.write(textwrap.dedent('''
        """Run the Boltz CLI with a narrow unpickling allowlist.

        torch>=2.6 loads checkpoints with weights_only=True, because a pickle
        can name any callable and have it invoked on load. Boltz-2's .ckpt
        embeds its training config as an omegaconf DictConfig, which the
        restricted unpickler refuses by default.

        We extend the allowlist by the specific config classes and nothing
        else. The restricted unpickler stays active for everything beyond
        them. This is deliberately NOT weights_only=False, which would
        disable the check entirely and permit arbitrary code execution.
        """
        import sys, typing, collections
        import torch.serialization as ts

        from omegaconf.dictconfig import DictConfig
        from omegaconf.listconfig import ListConfig
        from omegaconf.base import ContainerMetadata, Metadata
        from omegaconf.nodes import AnyNode, ValueNode

        ALLOWED = [
            DictConfig, ListConfig, ContainerMetadata, Metadata,
            AnyNode, ValueNode,
            typing.Any, collections.defaultdict,
            dict, list, int, str, float, bool,
        ]
        ts.add_safe_globals(ALLOWED)
        print(f"[boltz_run] allowlisted {len(ALLOWED)} classes for torch.load",
              file=sys.stderr)

        from boltz.main import cli
        sys.exit(cli())
    ''').lstrip())

print("wrapper:", WRAPPER)

# the shadowed sitecustomize does nothing — remove it rather than leave
# a dead global mutation behind
dead = "/usr/local/lib/python3.13/dist-packages/sitecustomize.py"
if os.path.exists(dead):
    os.remove(dead)
    print("removed:", dead)

wrapper: /content/tools/boltz_run.py
removed: /usr/local/lib/python3.13/dist-packages/sitecustomize.py


In [ ]:
# Checkpoint now loads. The run fails further in, on NVIDIA's optional
# cuequivariance kernels
import os, subprocess, time

PROJ = "/content/drive/MyDrive/il2_mimetic"
S5, IN, OUT = f"{PROJ}/stage5_complex", f"{PROJ}/stage5_complex/inputs", f"{PROJ}/stage5_complex/preds"
os.makedirs(OUT, exist_ok=True)

job = "ctrl_IL2_ternary"
fa  = f"{IN}/{job}.fasta"
assert os.path.exists(fa), fa

t0 = time.time()
p = subprocess.run(
    ["python","/content/tools/boltz_run.py","predict", fa,
     "--out_dir", OUT,
     "--output_format","pdb",
     "--seed","0",
     "--accelerator","gpu",
     "--use_msa_server",
     "--override"],
    capture_output=True, text=True)
dt = (time.time()-t0)/60

print(f"exit {p.returncode}   {dt:.1f} min")
print(p.stdout[-2500:])
if p.returncode != 0:
    print("---- stderr ----"); print(p.stderr[-3500:])

exit 1   1.0 min
MSA server enabled: https://api.colabfold.com
MSA server authentication: no credentials provided
Checking input data.
All inputs are already processed.
Processing 0 inputs with 0 threads.
Running structure prediction for 1 input.
Predicting                                      0/1 0:00:01 • -:--:-- 0.00it/s 

---- stderr ----
kages/boltz/model/models/boltz2.py", line 1059, in predict_step
    out = self(
        batch,
    ...<4 lines>...
        run_confidence_sequentially=True,
    )
  File "/usr/local/lib/python3.13/dist-packages/torch/nn/modules/module.py", line 1779, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
           ~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/torch/nn/modules/module.py", line 1790, in _call_impl

... [36 lines trimmed] ...
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/torch/nn/modules/module.py", line 

In [ ]:
PROJ = "/content/drive/MyDrive/il2_mimetic"
IN, OUT = f"{PROJ}/stage5_complex/inputs", f"{PROJ}/stage5_complex/preds"

job = "ctrl_IL2_ternary"
t0 = time.time()
p = subprocess.run(
    ["python","/content/tools/boltz_run.py","predict", f"{IN}/{job}.fasta",
     "--out_dir", OUT,
     "--output_format","pdb",
     "--seed","0",
     "--accelerator","gpu",
     "--use_msa_server",
     "--no_kernels",
     "--override"],
    capture_output=True, text=True)
dt = (time.time()-t0)/60

print(f"exit {p.returncode}   {dt:.1f} min")
print(p.stdout[-2500:])
if p.returncode != 0:
    print("---- stderr ----"); print(p.stderr[-4000:])

exit 0   2.5 min
MSA server enabled: https://api.colabfold.com
MSA server authentication: no credentials provided
Checking input data.
All inputs are already processed.
Processing 0 inputs with 0 threads.
Running structure prediction for 1 input.
Number of failed examples: 0
Predicting ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/1 0:01:29 • 0:00:00 0.00it/s 



In [ ]:
import os, glob, json
import numpy as np

PROJ = "/content/drive/MyDrive/il2_mimetic"
OUT  = f"{PROJ}/stage5_complex/preds"
job  = "ctrl_IL2_ternary"

pdbs = sorted(glob.glob(f"{OUT}/boltz_results_{job}/predictions/{job}/*.pdb"))
cjs  = sorted(glob.glob(f"{OUT}/boltz_results_{job}/predictions/{job}/confidence*.json"))
print("structures:", [os.path.basename(p) for p in pdbs])
print("confidence:", [os.path.basename(p) for p in cjs])

# --- chain composition ---
from collections import Counter, defaultdict
seen = defaultdict(set)
with open(pdbs[0]) as fh:
    for ln in fh:
        if ln.startswith("ATOM"):
            seen[ln[21]].add(int(ln[22:26]))
print("\nchain : residues")
for ch in sorted(seen):
    print(f"  {ch} : {len(seen[ch])}")
print("total:", sum(len(v) for v in seen.values()), "(expected 518)")

# --- confidence ---
c = json.load(open(cjs[0]))
print("\n--- confidence keys ---")
for k, v in c.items():
    if isinstance(v, (int, float)):
        print(f"  {k:28s} {v:.4f}")
    elif isinstance(v, list):
        print(f"  {k:28s} shape {np.array(v).shape}")

structures: ['ctrl_IL2_ternary_model_0.pdb']
confidence: ['confidence_ctrl_IL2_ternary_model_0.json']

chain : residues
  A : 133
  B : 214
  C : 199
total: 546 (expected 518)

--- confidence keys ---
  confidence_score             0.9262
  ptm                          0.9233
  iptm                         0.9082
  ligand_iptm                  0.0000
  protein_iptm                 0.9082
  complex_plddt                0.9307
  complex_iplddt               0.9467
  complex_pde                  0.4499
  complex_ipde                 0.7469


In [ ]:
PROJ = "/content/drive/MyDrive/il2_mimetic"
IN = f"{PROJ}/stage5_complex/inputs"

import glob, os
for fa in sorted(glob.glob(f"{IN}/*.fasta")):
    tot, parts = 0, []
    hdr = None
    with open(fa) as fh:
        for ln in fh:
            ln = ln.strip()
            if ln.startswith(">"):
                hdr = ln[1:]
            elif ln:
                parts.append(f"{hdr.split('|')[0]}={len(ln)}")
                tot += len(ln)
    print(f"{os.path.basename(fa)[:-6]:28s} {tot:4d}  {' '.join(parts)}")

ctrl_IL2_alpha                350  A=133 B=217
ctrl_IL2_ternary              546  A=133 B=214 C=199
ctrl_Neo_alpha                321  A=104 B=217
ctrl_Neo_ternary              517  A=104 B=214 C=199
ctrl_scram_alpha              322  A=105 B=217
ctrl_scram_ternary            518  A=105 B=214 C=199
pad0_len100_13_fixContacts_3_alpha  315  A=98 B=217
pad0_len100_13_fixContacts_3_ternary  511  A=98 B=214 C=199
pad0_len100_15_fixAll_4_alpha  322  A=105 B=217
pad0_len100_15_fixAll_4_ternary  518  A=105 B=214 C=199
pad0_len100_22_fixAll_0_alpha  324  A=107 B=217
pad0_len100_22_fixAll_0_ternary  520  A=107 B=214 C=199
pad0_len100_2_fixContacts_3_alpha  332  A=115 B=217
pad0_len100_2_fixContacts_3_ternary  528  A=115 B=214 C=199
pad0_len100_37_fixContacts_2_alpha  328  A=111 B=217
pad0_len100_37_fixContacts_2_ternary  524  A=111 B=214 C=199
pad2_len100_7_fixAll_3_alpha  308  A=91 B=217
pad2_len100_7_fixAll_3_ternary  504  A=91 B=214 C=199


In [ ]:
import subprocess
h = subprocess.run(["python","/content/tools/boltz_run.py","predict","--help"],
                   capture_output=True, text=True).stdout
for ln in h.splitlines():
    if any(k in ln.lower() for k in ["pae","pde","plddt","full","confidence"]):
        print(ln)

  --write_full_pae                Whether to dump the pae into a npz file.
  --write_full_pde                Whether to dump the pde into a npz file.


In [ ]:
# Batch run, resumable on output-file existence rather than on a record
# of what was attempted. The pre-PAE control run is detected by the
# absence of pae*.npz and cleared so it regenerates with the full matrix
import os, glob, shutil, subprocess, time

PROJ = "/content/drive/MyDrive/il2_mimetic"
IN   = f"{PROJ}/stage5_complex/inputs"
OUT  = f"{PROJ}/stage5_complex/preds"
LOG  = f"{PROJ}/stage5_complex/run_log.tsv"

# the first control ran before --write_full_pae; drop it so it regenerates
stale = f"{OUT}/boltz_results_ctrl_IL2_ternary"
if os.path.isdir(stale) and not glob.glob(f"{stale}/predictions/*/pae*.npz"):
    shutil.rmtree(stale); print("cleared stale control (no PAE matrix)")

jobs = sorted(os.path.basename(p)[:-6] for p in glob.glob(f"{IN}/*.fasta"))

def done(j):
    d = f"{OUT}/boltz_results_{j}/predictions/{j}"
    return bool(glob.glob(f"{d}/*.pdb")) and bool(glob.glob(f"{d}/pae*.npz"))

todo = [j for j in jobs if not done(j)]
print(f"{len(jobs)} jobs · {len(jobs)-len(todo)} complete · {len(todo)} to run")
print(f"rough estimate: {len(todo)*2.2:.0f} min\n")

if not os.path.exists(LOG):
    with open(LOG,"w") as fh: fh.write("job\texit\tminutes\n")

t_start = time.time()
for i, j in enumerate(todo, 1):
    t0 = time.time()
    p = subprocess.run(
        ["python","/content/tools/boltz_run.py","predict", f"{IN}/{j}.fasta",
         "--out_dir", OUT, "--output_format","pdb", "--seed","0",
         "--accelerator","gpu", "--use_msa_server", "--no_kernels",
         "--write_full_pae", "--write_full_pde"],
        capture_output=True, text=True)
    dt = (time.time()-t0)/60
    ok = "ok " if p.returncode == 0 else "FAIL"
    print(f"[{i:2d}/{len(todo)}] {ok} {j:38s} {dt:4.1f} min")
    if p.returncode != 0:
        tail = [l for l in p.stderr.strip().splitlines() if l.strip()][-1:]
        print(f"         {tail[0][:180] if tail else '(no stderr)'}")
    with open(LOG,"a") as fh:
        fh.write(f"{j}\t{p.returncode}\t{dt:.2f}\n")

print(f"\ntotal {(time.time()-t_start)/60:.1f} min · "
      f"complete {sum(done(j) for j in jobs)}/{len(jobs)}")

18 jobs · 1 complete · 17 to run
rough estimate: 37 min

[ 1/17] ok  ctrl_IL2_alpha                          2.4 min
[ 2/17] ok  ctrl_Neo_alpha                          0.2 min
[ 3/17] ok  ctrl_Neo_ternary                        4.1 min
[ 4/17] ok  ctrl_scram_alpha                       21.7 min
[ 5/17] ok  ctrl_scram_ternary                      2.4 min
[ 6/17] ok  pad0_len100_13_fixContacts_3_alpha      1.5 min
[ 7/17] ok  pad0_len100_13_fixContacts_3_ternary    2.4 min
[ 8/17] ok  pad0_len100_15_fixAll_4_alpha           1.5 min
[ 9/17] ok  pad0_len100_15_fixAll_4_ternary         2.4 min
[10/17] ok  pad0_len100_22_fixAll_0_alpha           1.5 min
[11/17] ok  pad0_len100_22_fixAll_0_ternary         2.4 min
[12/17] ok  pad0_len100_2_fixContacts_3_alpha       1.5 min
[13/17] ok  pad0_len100_2_fixContacts_3_ternary     2.4 min
[14/17] ok  pad0_len100_37_fixContacts_2_alpha      1.5 min
[15/17] ok  pad0_len100_37_fixContacts_2_ternary    2.4 min
[16/17] ok  pad2_len100_7_fixAll_3_alpha   

In [ ]:
# Completeness check against files on disk. This is not redundant with
# the batch log: one job exited 0, logged ok, and produced nothing
import glob, os
PROJ = "/content/drive/MyDrive/il2_mimetic"
IN, OUT = f"{PROJ}/stage5_complex/inputs", f"{PROJ}/stage5_complex/preds"

for fa in sorted(glob.glob(f"{IN}/*.fasta")):
    j = os.path.basename(fa)[:-6]
    d = f"{OUT}/boltz_results_{j}/predictions/{j}"
    pdb = glob.glob(f"{d}/*.pdb"); pae = glob.glob(f"{d}/pae*.npz")
    if not (pdb and pae):
        print(f"INCOMPLETE  {j:40s} pdb={len(pdb)} pae={len(pae)}")
        print("   files:", sorted(os.listdir(d)) if os.path.isdir(d) else "(no dir)")


INCOMPLETE  ctrl_Neo_alpha                           pdb=0 pae=0
   files: (no dir)


In [ ]:
# Diagnosis of the silent failure: the MSA server returned a corrupt
# archive, Boltz caught the error internally, printed "Skipping" and
# exited 0. An exit code cannot be a completeness check for a tool that
# handles its own errors
import os, glob, subprocess, time

PROJ = "/content/drive/MyDrive/il2_mimetic"
IN, OUT = f"{PROJ}/stage5_complex/inputs", f"{PROJ}/stage5_complex/preds"

# what does the input look like?
fa = f"{IN}/ctrl_Neo_alpha.fasta"
print(open(fa).read())
print("---- existing output dirs matching 'Neo' ----")
print([d for d in os.listdir(OUT) if "Neo" in d])

# re-run with override and show everything
t0 = time.time()
p = subprocess.run(
    ["python","/content/tools/boltz_run.py","predict", fa,
     "--out_dir", OUT, "--output_format","pdb", "--seed","0",
     "--accelerator","gpu", "--use_msa_server", "--no_kernels",
     "--write_full_pae", "--write_full_pde", "--override"],
    capture_output=True, text=True)
print(f"\nexit {p.returncode}   {(time.time()-t0)/60:.1f} min")
print("---- stdout ----"); print(p.stdout[-3000:])
print("---- stderr ----"); print(p.stderr[-3000:])

>A|protein|empty
GSHMPKKKIQLHAEHALYDALMILNIVKTNSPPAEEKLEDYAFNFELILEEIARLFESGDQKDEAEKAKRMKEWMKRIKTTASEDEQEEMANAIITILQSWIFS
>B|protein
ELCDDDPPEIPHATFKAMAYKEGTMLNCECKRGFRRIKSGSLYMLCTGNSSHSSWDNQCQCTSSATRQTTKQVTPQPEEQKERKTTEMQSPMQPVDQASLPGHCREPPPWENEATERIYHFVVGQMVYYQCVQGYRALHRGPAESVCKMTHGKTRWTQPQLICTGEMETSQFPGEEKPQASPEGRPESETSCLVTTTDFQIQTEMAATMETSIFTTE

---- existing output dirs matching 'Neo' ----
['boltz_results_ctrl_Neo_alpha', 'boltz_results_ctrl_Neo_ternary']

exit 0   0.2 min
---- stdout ----
MSA server enabled: https://api.colabfold.com
MSA server authentication: no credentials provided
Checking input data.
Found 0 existing processed inputs, skipping them.
Processing 1 inputs with 1 threads.
Found explicit empty MSA for some proteins, will run these in single sequence mode. Keep in mind that the model predictions will be suboptimal without an MSA.
Generating MSA for /content/drive/MyDrive/il2_mimetic/stage5_complex/inputs/ctrl_Neo_alpha.fasta with 1 protein entities.
Calling MSA ser

In [ ]:
# Retry with a bounded attempt loop, distinguishing a transient server
# fault from a real one by whether the ReadError recurs. Succeeded on the
# first attempt, confirming the failure was transient
import os, shutil, subprocess, time, glob

PROJ = "/content/drive/MyDrive/il2_mimetic"
IN, OUT = f"{PROJ}/stage5_complex/inputs", f"{PROJ}/stage5_complex/preds"
job = "ctrl_Neo_alpha"

d = f"{OUT}/boltz_results_{job}"
if os.path.isdir(d):
    shutil.rmtree(d); print("cleared", d)

for attempt in (1, 2, 3):
    t0 = time.time()
    p = subprocess.run(
        ["python","/content/tools/boltz_run.py","predict", f"{IN}/{job}.fasta",
         "--out_dir", OUT, "--output_format","pdb", "--seed","0",
         "--accelerator","gpu", "--use_msa_server", "--no_kernels",
         "--write_full_pae", "--write_full_pde", "--override"],
        capture_output=True, text=True)
    ok = bool(glob.glob(f"{d}/predictions/{job}/*.pdb"))
    print(f"attempt {attempt}: exit {p.returncode}  {(time.time()-t0)/60:.1f} min  "
          f"output={'yes' if ok else 'NO'}")
    if ok: break
    if "ReadError" in p.stdout or "ReadError" in p.stderr:
        print("   MSA server returned a corrupt archive again; waiting 60s")
        time.sleep(60)
    else:
        print(p.stderr[-1500:]); break

cleared /content/drive/MyDrive/il2_mimetic/stage5_complex/preds/boltz_results_ctrl_Neo_alpha
attempt 1: exit 0  1.5 min  output=yes


In [ ]:
# Decompose the PAE matrix into per-chain-pair interface blocks, averaging
# the two off-diagonal blocks to symmetrise an inherently asymmetric
# quantity. The beta-gc pair is native in every ternary run regardless of
# ligand, so its near-constant value is a positive control on the
# decomposition itself
import os, glob, json
import numpy as np, pandas as pd

PROJ = "/content/drive/MyDrive/il2_mimetic"
IN, OUT, S5 = (f"{PROJ}/stage5_complex/inputs", f"{PROJ}/stage5_complex/preds",
               f"{PROJ}/stage5_complex")

def chains_of(job):
    out, hdr = [], None
    for ln in open(f"{IN}/{job}.fasta"):
        ln = ln.strip()
        if ln.startswith(">"): hdr = ln[1:].split("|")[0]
        elif ln: out.append((hdr, len(ln)))
    return out

def blocks(job):
    i, rng = 0, {}
    for ch, n in chains_of(job):
        rng[ch] = (i, i+n); i += n
    return rng, i

def load_pae(job):
    f = glob.glob(f"{OUT}/boltz_results_{job}/predictions/{job}/pae*.npz")
    if not f: return None
    z = np.load(f[0]); return np.squeeze(z[list(z.keys())[0]])

def iface(pae, r1, r2):
    a = pae[r1[0]:r1[1], r2[0]:r2[1]]; b = pae[r2[0]:r2[1], r1[0]:r1[1]]
    return float((a.mean() + b.mean()) / 2)

rows = []
for fa in sorted(glob.glob(f"{IN}/*.fasta")):
    job = os.path.basename(fa)[:-6]
    d = f"{OUT}/boltz_results_{job}/predictions/{job}"
    cj = glob.glob(f"{d}/confidence*.json")
    if not cj: print("MISSING:", job); continue
    c = json.load(open(cj[0]))
    rng, n = blocks(job); pae = load_pae(job)
    kind = "alpha" if job.endswith("_alpha") else "ternary"
    r = dict(job=job, design=job.rsplit("_",1)[0], kind=kind, n_res=n,
             iptm=c.get("iptm"), ptm=c.get("ptm"),
             plddt=c.get("complex_plddt"), iplddt=c.get("complex_iplddt"))
    if pae is not None:
        r["pae_shape"] = pae.shape[0]
        ch = list(rng)
        if kind == "ternary":
            r["pae_d_beta"] = iface(pae, rng[ch[0]], rng[ch[1]])
            r["pae_d_gc"]   = iface(pae, rng[ch[0]], rng[ch[2]])
            r["pae_beta_gc"]= iface(pae, rng[ch[1]], rng[ch[2]])
        else:
            r["pae_d_alpha"]= iface(pae, rng[ch[0]], rng[ch[1]])
    rows.append(r)

df = pd.DataFrame(rows)
bad = df[df.pae_shape.notna() & (df.pae_shape != df.n_res)]
print("PAE/length mismatch:", "NONE" if not len(bad) else bad.job.tolist())

# --- β·γc is native in every ternary run: it should be near-constant ---
t = df[df.kind=="ternary"]
print(f"\nβ·γc interface PAE across {len(t)} ternary runs: "
      f"{t.pae_beta_gc.mean():.2f} ± {t.pae_beta_gc.std():.2f} "
      f"(range {t.pae_beta_gc.min():.2f}–{t.pae_beta_gc.max():.2f})")

# --- one row per design ---
p = df.pivot_table(index="design", columns="kind",
                   values=["iptm","plddt"], aggfunc="first")
p.columns = [f"{a}_{b}" for a, b in p.columns]
extra = df.set_index("design")[["pae_d_beta","pae_d_gc","pae_d_alpha"]].groupby(level=0).first()
tab = p.join(extra)
tab["iptm_contrast"]  = tab.iptm_ternary - tab.iptm_alpha
tab["pae_contrast"]   = tab.pae_d_alpha - tab[["pae_d_beta","pae_d_gc"]].mean(axis=1)

order = (["ctrl_IL2","ctrl_Neo","ctrl_scram"] +
         sorted(d for d in tab.index if not d.startswith("ctrl_")))
tab = tab.reindex([d for d in order if d in tab.index])

pd.set_option("display.width", 200)
print("\n" + tab.round(3).to_string())
tab.round(4).to_csv(f"{S5}/stage5_metrics.tsv", sep="\t")

PAE/length mismatch: NONE

β·γc interface PAE across 9 ternary runs: 7.98 ± 0.51 (range 6.95–8.62)

                              iptm_alpha  iptm_ternary  plddt_alpha  plddt_ternary  pae_d_beta  pae_d_gc  pae_d_alpha  iptm_contrast  pae_contrast
design                                                                                                                                            
ctrl_IL2                           0.884         0.908        0.733          0.931       6.715     6.654       10.177          0.024         3.492
ctrl_Neo                           0.761         0.891        0.716          0.933       6.626     6.670       14.310          0.130         7.663
ctrl_scram                         0.706         0.870        0.646          0.834       8.878     9.062       16.535          0.164         7.565
pad0_len100_13_fixContacts_3       0.815         0.896        0.763          0.943       5.668     6.016       12.442          0.081         6.599
pad0_len100_15_fix

In [ ]:
# Single contact check
import glob
from collections import defaultdict
import numpy as np

job = "pad0_len100_2_fixContacts_3_alpha"
p = glob.glob(f"{OUT}/boltz_results_{job}/predictions/{job}/*.pdb")[0]
xyz = defaultdict(list)
for ln in open(p):
    if ln.startswith("ATOM") and ln[12:16].strip() == "CA":
        xyz[ln[21]].append([float(ln[30+8*i:38+8*i]) for i in range(3)])
A, B = np.array(xyz["A"]), np.array(xyz["B"])
d = np.linalg.norm(A[:,None,:] - B[None,:,:], axis=-1)
print(f"min Cα–Cα {d.min():.1f} Å · contacts <8 Å: {(d<8).sum()}")

min Cα–Cα 5.1 Å · contacts <8 Å: 18


In [ ]:
# All contacts check
import glob
from collections import defaultdict
import numpy as np, pandas as pd

def contacts(job):
    f = glob.glob(f"{OUT}/boltz_results_{job}/predictions/{job}/*.pdb")
    if not f: return None
    xyz = defaultdict(list)
    for ln in open(f[0]):
        if ln.startswith("ATOM") and ln[12:16].strip() == "CA":
            xyz[ln[21]].append([float(ln[30+8*i:38+8*i]) for i in range(3)])
    A, B = np.array(xyz["A"]), np.array(xyz["B"])
    d = np.linalg.norm(A[:,None,:] - B[None,:,:], axis=-1)
    return dict(job=job, min_d=d.min(), c8=(d<8).sum(), c12=(d<12).sum(),
                n_A=len(A), n_B=len(B))

rows = [contacts(os.path.basename(f)[:-6])
        for f in sorted(glob.glob(f"{IN}/*_alpha.fasta"))]
cd = pd.DataFrame([r for r in rows if r])
cd["design"] = cd.job.str.replace("_alpha", "", regex=False)
cd = cd.merge(tab[["pae_d_alpha","iptm_alpha"]], left_on="design",
              right_index=True, how="left")
order = ["ctrl_IL2","ctrl_Neo","ctrl_scram"] + \
        sorted(d for d in cd.design if not d.startswith("ctrl_"))
print(cd.set_index("design").reindex(order)
        [["min_d","c8","c12","pae_d_alpha","iptm_alpha"]].round(2).to_string())

                              min_d  c8  c12  pae_d_alpha  iptm_alpha
design                                                               
ctrl_IL2                       4.78  30  269        10.18        0.88
ctrl_Neo                       6.05  14  255        14.31        0.76
ctrl_scram                     3.92  22  244        16.54        0.71
pad0_len100_13_fixContacts_3   4.79  39  330        12.44        0.82
pad0_len100_15_fixAll_4        4.96  44  401        21.58        0.42
pad0_len100_22_fixAll_0        4.27  39  357        11.17        0.87
pad0_len100_2_fixContacts_3    5.13  18  246        20.64        0.44
pad0_len100_37_fixContacts_2   4.25  46  372        12.48        0.81
pad2_len100_7_fixAll_3         4.53  31  319        11.02        0.83


#### Results

1. Boltz-2 required four successive fixes before producing output: an unpinned dependency install with the Python bound overridden, a narrowed unpickling allowlist delivered through a wrapper rather than `sitecustomize`, `--no_kernels` to bypass optional NVIDIA kernels, and `--write_full_pae` to emit the matrix the analysis needs. 18 jobs completed in 54 minutes.

2. The pre-registered primary metric failed its negative control.
| | ipTM ternary | ipTM alpha | ipTM contrast |
|---|---|---|---|
| native IL-2 | 0.908 | 0.884 | 0.024 |
| Neo-2/15 | 0.891 | 0.761 | 0.130 |
| scrambled | 0.870 | 0.706 | 0.164 |

    The scrambled sequence is predicted against the βγc receptor assembly at ipTM 0.870, against native IL-2's 0.908. More damagingly, its selectivity contrast of 0.164 exceeds Neo-2/15's 0.130. The metric ranks a shuffled sequence as more selective than the published, experimentally validated reference.

    Native IL-2's contrast of 0.024 is correct and interpretable: IL-2 binds both assemblies, so it should show no preference. The metric reads the positive control correctly and fails on the negative one. ipTM measures whether the model can place two chains together with confidence, and for a compact helical bundle against a large receptor surface it can do so almost regardless of sequence identity.
3. Decomposing the predicted aligned error matrix into chain-pair blocks gives a metric that separates the controls in the correct order on the target interface:

    | | PAE design·β | PAE design·γc | mean target | PAE design·α |
    |---|---|---|---|---|
    | native IL-2 | 6.72 | 6.65 | 6.69 | 10.18 |
    | Neo-2/15 | 6.63 | 6.67 | 6.65 | 14.31 |
    | scrambled | 8.88 | 9.06 | 8.97 | 16.54 |

    The scrambled control is  separated from both real molecules on target-interface PAE, where ipTM failed to provide.

4. Five of six designs are predicted to engage the βγc assembly more confidently than Neo-2/15. This comparison is like-for-like: Neo-2/15, the scrambled control and all designs were predicted in single-sequence mode. Native IL-2 is the only ligand that received an MSA, so its value carries an advantage the others do not and it is the weaker of the two references. Lower PAE for an untested design than for a validated one should be read with suspicion rather than satisfaction. Confidence metrics reward idealised structures, and de novo designs are more idealised than natural proteins. The same effect seen in Stage 3, where Neo-2/15 outscored native IL-2 on ProteinMPNN designability.

5. α-receptor engagement
| design | ipTM α | PAE design·α | contacts < 8 Å | contacts < 12 Å |
|---|---|---|---|---|
| Neo-2/15 | 0.761 | 14.31 | **14** | 255 |
| pad0_len100_2_fixContacts_3 | 0.441 | 20.64 | **18** | 246 |
| scrambled | 0.706 | 16.54 | 22 | 244 |
| native IL-2 | 0.884 | 10.18 | 30 | 269 |
| pad2_len100_7_fixAll_3 | 0.830 | 11.02 | 31 | 319 |
| pad0_len100_13_fixContacts_3 | 0.815 | 12.44 | 39 | 330 |
| pad0_len100_22_fixAll_0 | 0.866 | 11.17 | 39 | 357 |
| pad0_len100_15_fixAll_4 | 0.418 | 21.58 | 44 | 401 |
| pad0_len100_37_fixContacts_2 | 0.814 | 12.48 | **46** | 372 |

    Neo-2/15 makes the fewest α contacts in the panel, as a selective molecule should. Native IL-2 makes 30, consistent with genuinely binding α.

6. Selecting 6 of 12 candidates by greedy max-min on TM-score reduced maximum pairwise similarity from 0.979 to 0.921 but left the median unchanged, at 0.891 against 0.892. Since TM > 0.5 indicates the same fold and 0.89 indicates near-identical structure, the twelve Stage 4 candidates are topologically homogeneous. The selection removed near-duplicates and could not manufacture diversity that was not present. This is a property of the pipeline, not of the selection: a fixed three-segment motif with a narrow length budget and a single RFdiffusion checkpoint produces one dominant topology. Any conclusion drawn from this panel concerns one fold sampled six times, not six independent solutions to the design problem.


#### Conclusion
1. The pre-registered primary metric was falsified by its negative control, and
replaced. ipTM contrast ranked a composition-matched scrambled sequence as more selective than Neo-2/15. The metric was specified before any prediction was run and discarded on the evidence of the control, not on the evidence of inconvenient design scores. Both metrics and all values are reported.

2. Running a scrambled negative control is what made this stage interpretable. Without it, the design ipTM contrasts of 0.03-0.45 would have looked like a selectivity ranking. With it, the ranking is known to be unusable. The control was four of eighteen jobs and it determined what the other fourteen mean.

3. The informative quantity is raw target-interface PAE, not a contrast between panels. Decomposing the PAE matrix per chain pair separates the scrambled control from both real molecules on the target interface. The difference between panels does not: Neo-2/15 and scrambled give 7.66 and 7.57. A selectivity metric built from two high confidence estimates inherits the uninformativeness of both.

4. Five of six designs are predicted to engage βγc more confidently than Neo-2 15 on a like-for-like single-sequence basis. Confidence metrics reward idealised structures and de novo designs are idealised, so this is as likely to reflect that bias as it is to reflect better binding. The same pattern appeared in Stage 3 on an unrelated metric.

5. Four of six designs make more α contacts than native IL-2, which binds α. The design with the strongest target engagement makes the most α contacts of anything in the panel.

6. Low confidence is ambiguous, so geometry solved the ambiguity. Two designs share an α ipTM near 0.42. One has the fewest contacts in the panel and is genuinely disengaged, and the other has the most and is a confused prediction. Adding a contact count separated them, and no confidence metric would have.


#### Limitations
1. One seed, one diffusion sample per job.
2. The scrambled control is a single sequence at a single seed.
3. Six candidates, one topology.
4. No binding energy.
5. The negative control is a shuffled sequence, which does not fold. Clearing its interface PAE therefore establishes a proteinness floor, not a binding floor. The missing control is a folded chain with no epitope,design 37 with the motif mutated to alanine.